In [1]:
# ============================================================
# 09_AURORA_validation_optimized_regime_templates.ipynb
# AURORA-TWETF Validation-Optimized Regime Templates
#
# Purpose:
# 1. Load purged walk-forward probability outputs from Notebook 07.
# 2. Use validation folds only to optimize regime-to-weight templates.
# 3. Apply the selected templates to test folds without test leakage.
# 4. Compare optimized AURORA templates against:
#    - hand-crafted AURORA templates
#    - passive ETF baselines
#    - dynamic financial baselines from Notebook 08
# 5. Save paper-ready tables, figures, diagnostics, validation report,
#    and SHA-256 manifest.
#
# Important:
# - This notebook does not optimize on test data.
# - For each walk-forward fold, templates are optimized only on that fold's validation dates.
# - The optimized fold-specific template is then evaluated on that fold's test dates.
# - Educational/research backtest only.
# - Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Paths and run configuration
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

# Existing successful runs.
NOTEBOOK07_RUN_ID = "20260624_031817"
NOTEBOOK08_RUN_ID = "20260624_034204"
NOTEBOOK08B_RUN_ID = "20260624_070827"

NOTEBOOK07_ROOT = OUTPUT_ROOT / "purged_walk_forward_models" / f"run_{NOTEBOOK07_RUN_ID}"
NOTEBOOK08_ROOT = OUTPUT_ROOT / "stronger_financial_baselines_purged_allocation" / f"run_{NOTEBOOK08_RUN_ID}"
NOTEBOOK08B_ROOT = OUTPUT_ROOT / "aligned_oos_reanalysis" / f"run_{NOTEBOOK08B_RUN_ID}"

NOTEBOOK08_INPUT_INDEX = NOTEBOOK07_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv"
NOTEBOOK08_WEIGHT_DIR = NOTEBOOK08_ROOT / "weights"
NOTEBOOK08_POLICY_DESCRIPTION_PATH = NOTEBOOK08_ROOT / "tables" / "policy_descriptions.csv"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "validation_optimized_regime_templates" / f"run_{RUN_ID}"

WEIGHT_DIR = RUN_ROOT / "weights"
RETURN_DIR = RUN_ROOT / "returns"
TEMPLATE_DIR = RUN_ROOT / "templates"
PLOT_DIR = RUN_ROOT / "plots"
TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
DIAGNOSTIC_DIR = RUN_ROOT / "diagnostics"
PAPER_FIGURE_DIR = RUN_ROOT / "paper_figures"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    RUN_ROOT,
    WEIGHT_DIR,
    RETURN_DIR,
    TEMPLATE_DIR,
    PLOT_DIR,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    DIAGNOSTIC_DIR,
    PAPER_FIGURE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Notebook 09: Validation-Optimized Regime Templates")
print("=" * 80)
print("Timestamp UTC       :", RUN_TIMESTAMP)
print("Run ID              :", RUN_ID)
print("Notebook 07 root    :", NOTEBOOK07_ROOT)
print("Notebook 08 root    :", NOTEBOOK08_ROOT)
print("Notebook 08B root   :", NOTEBOOK08B_ROOT)
print("Notebook 08 registry:", NOTEBOOK08_INPUT_INDEX)
print("Run root            :", RUN_ROOT)
print("=" * 80)

for required_path in [
    NOTEBOOK07_ROOT,
    NOTEBOOK08_ROOT,
    NOTEBOOK08_INPUT_INDEX,
    NOTEBOOK08_WEIGHT_DIR,
    NOTEBOOK08_POLICY_DESCRIPTION_PATH,
]:
    if not Path(required_path).exists():
        raise FileNotFoundError(f"Required path not found: {required_path}")

# ============================================================
# 2. Global settings
# ============================================================

ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "CASH"
ALL_ASSETS = ETF_UNIVERSE + [CASH_COL]

CLASS_LABELS = [0, 1, 2, 3, 4]

REGIME_LABELS = {
    0: "Strong Bear",
    1: "Bear",
    2: "Neutral",
    3: "Bull",
    4: "Strong Bull",
}

ANNUALIZATION_DAYS = 252
INITIAL_CAPITAL = 1.0

TRANSACTION_COST_RATE = 0.0010
REBALANCE_FREQUENCY = "monthly"

RANDOM_STATE = 42
N_RANDOM_TEMPLATES = 1500
N_ELITE_TEMPLATES = 50

# Optimization objective weights.
OBJECTIVE_WEIGHTS = {
    "annual_return": 1.00,
    "annual_volatility": -0.20,
    "max_drawdown_abs": -0.30,
    "turnover": -0.05,
}

# Candidate allocation configurations.
OPTIMIZATION_CONFIGS = [
    {
        "strategy_name": "VOT_A_baseline_60_40_cash20",
        "alpha_20d": 0.60,
        "alpha_60d": 0.40,
        "uncertainty_cash_boost_max": 0.20,
        "max_etf_weight": 0.45,
        "max_00881_weight": 0.35,
        "max_cash_weight": 0.50,
        "min_cash_weight": 0.00,
        "min_confidence_for_full_risk": 0.55,
        "max_confidence_for_min_risk": 0.20,
    },
    {
        "strategy_name": "VOT_B_more60_30_70_cash20",
        "alpha_20d": 0.30,
        "alpha_60d": 0.70,
        "uncertainty_cash_boost_max": 0.20,
        "max_etf_weight": 0.45,
        "max_00881_weight": 0.35,
        "max_cash_weight": 0.50,
        "min_cash_weight": 0.00,
        "min_confidence_for_full_risk": 0.55,
        "max_confidence_for_min_risk": 0.20,
    },
    {
        "strategy_name": "VOT_C_more60_30_70_cash10",
        "alpha_20d": 0.30,
        "alpha_60d": 0.70,
        "uncertainty_cash_boost_max": 0.10,
        "max_etf_weight": 0.45,
        "max_00881_weight": 0.35,
        "max_cash_weight": 0.50,
        "min_cash_weight": 0.00,
        "min_confidence_for_full_risk": 0.55,
        "max_confidence_for_min_risk": 0.20,
    },
    {
        "strategy_name": "VOT_D_more60_30_70_no_cash_boost",
        "alpha_20d": 0.30,
        "alpha_60d": 0.70,
        "uncertainty_cash_boost_max": 0.00,
        "max_etf_weight": 0.45,
        "max_00881_weight": 0.35,
        "max_cash_weight": 0.50,
        "min_cash_weight": 0.00,
        "min_confidence_for_full_risk": 0.55,
        "max_confidence_for_min_risk": 0.20,
    },
]

# Hand-crafted templates from earlier notebooks.
BASELINE_CLASS_WEIGHT_TEMPLATES = {
    0: {"0050": 0.15, "006208": 0.25, "00692": 0.25, "00881": 0.00, "CASH": 0.35},
    1: {"0050": 0.25, "006208": 0.30, "00692": 0.30, "00881": 0.05, "CASH": 0.10},
    2: {"0050": 0.30, "006208": 0.30, "00692": 0.25, "00881": 0.15, "CASH": 0.00},
    3: {"0050": 0.30, "006208": 0.25, "00692": 0.20, "00881": 0.25, "CASH": 0.00},
    4: {"0050": 0.25, "006208": 0.20, "00692": 0.15, "00881": 0.40, "CASH": 0.00},
}

AGGRESSIVE_CLASS_WEIGHT_TEMPLATES = {
    0: {"0050": 0.20, "006208": 0.30, "00692": 0.25, "00881": 0.05, "CASH": 0.20},
    1: {"0050": 0.28, "006208": 0.30, "00692": 0.27, "00881": 0.10, "CASH": 0.05},
    2: {"0050": 0.30, "006208": 0.28, "00692": 0.22, "00881": 0.20, "CASH": 0.00},
    3: {"0050": 0.28, "006208": 0.22, "00692": 0.15, "00881": 0.35, "CASH": 0.00},
    4: {"0050": 0.22, "006208": 0.18, "00692": 0.10, "00881": 0.50, "CASH": 0.00},
}

DEFENSIVE_CLASS_WEIGHT_TEMPLATES = {
    0: {"0050": 0.10, "006208": 0.20, "00692": 0.25, "00881": 0.00, "CASH": 0.45},
    1: {"0050": 0.20, "006208": 0.25, "00692": 0.30, "00881": 0.00, "CASH": 0.25},
    2: {"0050": 0.30, "006208": 0.30, "00692": 0.25, "00881": 0.05, "CASH": 0.10},
    3: {"0050": 0.32, "006208": 0.28, "00692": 0.25, "00881": 0.15, "CASH": 0.00},
    4: {"0050": 0.30, "006208": 0.25, "00692": 0.20, "00881": 0.25, "CASH": 0.00},
}

SEED_TEMPLATES = {
    "manual_baseline": BASELINE_CLASS_WEIGHT_TEMPLATES,
    "manual_aggressive": AGGRESSIVE_CLASS_WEIGHT_TEMPLATES,
    "manual_defensive": DEFENSIVE_CLASS_WEIGHT_TEMPLATES,
}

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
    )

def clean_symbol_name(x):
    x = str(x)
    x = x.replace(".TW", "")
    x = x.replace(".TWO", "")
    x = x.replace("TW_", "")
    return x

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def read_table_auto(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            pass

    df.index.name = "date"
    return df.sort_index()

def load_etf_return_panel():
    candidates = [
        PANEL_DIR / "AURORA_etf_return_panel.parquet",
        PANEL_DIR / "AURORA_etf_returns_panel.parquet",
        PANEL_DIR / "AURORA_return_panel.parquet",
        MODELING_DIR / "AURORA_etf_return_panel.parquet",
    ]

    path = find_first_existing(candidates)

    if path is None:
        raise FileNotFoundError(
            "Could not find ETF return panel. Tried:\n"
            + "\n".join(str(p) for p in candidates)
        )

    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    df = df.sort_index()
    df = df.rename(columns={c: clean_symbol_name(c) for c in df.columns})

    missing = [s for s in ETF_UNIVERSE if s not in df.columns]
    if missing:
        raise ValueError(
            f"ETF return panel found at {path}, but missing ETF columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    df = df[ETF_UNIVERSE].copy()
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return df, path

def normalize_rows(df):
    out = df.copy()
    out = out.reindex(columns=ALL_ASSETS).fillna(0.0)
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out[out < 0] = 0.0

    row_sums = out.sum(axis=1)
    zero_mask = row_sums <= 0

    if zero_mask.any():
        out.loc[zero_mask, ETF_UNIVERSE] = 1.0 / len(ETF_UNIVERSE)
        out.loc[zero_mask, CASH_COL] = 0.0
        row_sums = out.sum(axis=1)

    out = out.div(row_sums, axis=0)
    return out

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    return p / row_sums

def proba_cols():
    return [f"proba_class_{i}" for i in CLASS_LABELS]

def probability_features(proba_df):
    cols = proba_cols()
    missing = [c for c in cols if c not in proba_df.columns]
    if missing:
        raise ValueError(f"Missing probability columns: {missing}")

    p = normalize_proba(proba_df[cols].values)

    class_values = np.asarray(CLASS_LABELS, dtype=float)
    expected_class = p @ class_values
    entropy = -np.sum(np.clip(p, 1e-12, 1.0) * np.log(np.clip(p, 1e-12, 1.0)), axis=1)
    normalized_entropy = entropy / np.log(len(CLASS_LABELS))
    sorted_p = np.sort(p, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    ordinal_variance = (p @ (class_values ** 2)) - expected_class ** 2
    confidence = 1.0 - normalized_entropy

    out = pd.DataFrame(index=proba_df.index)
    out["expected_class"] = expected_class
    out["entropy"] = entropy
    out["normalized_entropy"] = normalized_entropy
    out["confidence_score"] = confidence
    out["probability_margin"] = margin
    out["ordinal_variance"] = ordinal_variance
    out["p_bearish"] = p[:, 0] + p[:, 1]
    out["p_neutral"] = p[:, 2]
    out["p_bullish"] = p[:, 3] + p[:, 4]

    for i, label in enumerate(CLASS_LABELS):
        out[f"proba_class_{label}"] = p[:, i]

    return out

def latest_fold_deduplicate(proba_df, split_filter=None):
    df = proba_df.copy()

    if split_filter is not None and "split" in df.columns:
        if isinstance(split_filter, str):
            split_filter = [split_filter]
        df = df[df["split"].isin(split_filter)].copy()

    if df.empty:
        return df

    df = df.reset_index()

    if "fold_id" not in df.columns:
        df["fold_id"] = "WF0"

    df["fold_number"] = (
        df["fold_id"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .fillna("0")
        .astype(int)
    )

    split_priority = {"train": 0, "validation": 1, "test": 2}
    if "split" in df.columns:
        df["split_priority"] = df["split"].map(split_priority).fillna(0).astype(int)
    else:
        df["split_priority"] = 0

    df = df.sort_values(["date", "split_priority", "fold_number"])
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.set_index("date").sort_index()
    df.index.name = "date"

    return df.drop(columns=["fold_number", "split_priority"], errors="ignore")

# ============================================================
# 4. Backtest functions
# ============================================================

def get_rebalance_dates(index, frequency):
    idx = pd.DatetimeIndex(index).sort_values()

    if frequency == "monthly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.month])
    elif frequency == "quarterly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.quarter])
    elif frequency == "weekly":
        iso = idx.isocalendar()
        groups = pd.Series(idx, index=idx).groupby([iso.year, iso.week])
    else:
        raise ValueError(f"Unsupported rebalance frequency: {frequency}")

    dates = []
    for _, values in groups:
        dates.append(values.iloc[0])

    return pd.DatetimeIndex(dates)

def expand_rebalance_weights_to_daily(signal_weight_df, daily_index, rebalance_dates):
    signal = signal_weight_df.reindex(columns=ALL_ASSETS).fillna(0.0).copy()
    signal = normalize_rows(signal)
    signal_idx = pd.DatetimeIndex(signal.index).sort_values()

    daily = pd.DataFrame(index=daily_index, columns=ALL_ASSETS, dtype=float)

    for i, reb_date in enumerate(rebalance_dates):
        if i + 1 < len(rebalance_dates):
            period_idx = daily_index[(daily_index >= reb_date) & (daily_index < rebalance_dates[i + 1])]
        else:
            period_idx = daily_index[daily_index >= reb_date]

        prior_signals = signal_idx[signal_idx < reb_date]

        if len(prior_signals) == 0:
            signal_date = signal_idx[0]
        else:
            signal_date = prior_signals[-1]

        daily.loc[period_idx, ALL_ASSETS] = signal.loc[signal_date, ALL_ASSETS].values

    daily = daily.ffill().bfill()
    daily = normalize_rows(daily)

    return daily

def compute_turnover(daily_weights, rebalance_dates):
    turnover = pd.Series(0.0, index=daily_weights.index)
    prev_w = None

    for dt in rebalance_dates:
        if dt not in daily_weights.index:
            continue

        w = daily_weights.loc[dt, ALL_ASSETS]

        if prev_w is None:
            turnover.loc[dt] = w.drop(labels=[CASH_COL], errors="ignore").abs().sum()
        else:
            turnover.loc[dt] = (w - prev_w).abs().sum() / 2.0

        prev_w = w

    return turnover

def backtest_on_fixed_index(policy_name, signal_weights, etf_returns, evaluation_index):
    evaluation_index = pd.DatetimeIndex(evaluation_index).sort_values()

    returns = etf_returns.copy()
    returns[CASH_COL] = 0.0
    returns = returns.reindex(evaluation_index)

    if returns[ALL_ASSETS].isna().any().any():
        missing_rows = returns[returns[ALL_ASSETS].isna().any(axis=1)]
        raise ValueError(
            f"Return panel has missing rows for {policy_name}. "
            f"Example missing dates: {missing_rows.index[:5].tolist()}"
        )

    signal = signal_weights.copy()
    signal.index = pd.to_datetime(signal.index)
    signal = signal.sort_index()
    signal = signal.reindex(columns=ALL_ASSETS).fillna(0.0)

    signal_aligned = signal.reindex(evaluation_index).ffill().bfill()
    signal_aligned = normalize_rows(signal_aligned)

    rebalance_dates = get_rebalance_dates(evaluation_index, REBALANCE_FREQUENCY)

    daily_weights = expand_rebalance_weights_to_daily(
        signal_weight_df=signal_aligned,
        daily_index=evaluation_index,
        rebalance_dates=rebalance_dates,
    )

    gross_return = (daily_weights[ALL_ASSETS] * returns[ALL_ASSETS]).sum(axis=1)
    turnover = compute_turnover(daily_weights, rebalance_dates)
    transaction_cost = turnover * TRANSACTION_COST_RATE
    net_return = gross_return - transaction_cost

    equity = (1.0 + net_return).cumprod() * INITIAL_CAPITAL

    out = pd.DataFrame(index=evaluation_index)
    out.index.name = "date"
    out["policy_name"] = policy_name
    out["gross_return"] = gross_return
    out["turnover"] = turnover
    out["transaction_cost"] = transaction_cost
    out["net_return"] = net_return
    out["equity"] = equity
    out["drawdown"] = equity / equity.cummax() - 1.0
    out["is_rebalance_date"] = out.index.isin(rebalance_dates)

    return out, daily_weights

def performance_metrics(return_df):
    r = return_df["net_return"].astype(float).copy()
    equity = return_df["equity"].astype(float).copy()
    drawdown = return_df["drawdown"].astype(float).copy()

    n = len(r)
    if n == 0:
        return {}

    total_return = float(equity.iloc[-1] / equity.iloc[0] - 1.0) if equity.iloc[0] != 0 else np.nan
    annual_return = float((1.0 + total_return) ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)

    annual_vol = float(r.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if n > 1 else np.nan
    sharpe = annual_return / annual_vol if annual_vol and annual_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if len(downside) > 1 else np.nan
    sortino = annual_return / downside_vol if downside_vol and downside_vol > 0 else np.nan

    max_drawdown = float(drawdown.min())
    calmar = annual_return / abs(max_drawdown) if max_drawdown < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()),
        "end_date": str(r.index.max().date()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "hit_rate": float((r > 0).mean()),
        "avg_daily_return": float(r.mean()),
        "avg_turnover": float(return_df["turnover"].mean()),
        "total_turnover": float(return_df["turnover"].sum()),
        "total_transaction_cost": float(return_df["transaction_cost"].sum()),
        "final_equity": float(equity.iloc[-1]),
    }

def objective_from_metrics(metrics):
    annual_return = float(metrics.get("annual_return", 0.0))
    annual_vol = float(metrics.get("annual_volatility", 0.0))
    max_drawdown_abs = abs(float(metrics.get("max_drawdown", 0.0)))
    turnover = float(metrics.get("total_turnover", 0.0))

    objective = (
        OBJECTIVE_WEIGHTS["annual_return"] * annual_return
        + OBJECTIVE_WEIGHTS["annual_volatility"] * annual_vol
        + OBJECTIVE_WEIGHTS["max_drawdown_abs"] * max_drawdown_abs
        + OBJECTIVE_WEIGHTS["turnover"] * turnover
    )

    return float(objective)

def add_composite_rank(df):
    out = df.copy()

    out["rank_total_return"] = out["total_return"].rank(ascending=False, method="min")
    out["rank_sharpe"] = out["sharpe_ratio"].rank(ascending=False, method="min")
    out["rank_sortino"] = out["sortino_ratio"].rank(ascending=False, method="min")
    out["rank_drawdown"] = out["max_drawdown"].rank(ascending=False, method="min")
    out["rank_calmar"] = out["calmar_ratio"].rank(ascending=False, method="min")

    out["allocation_composite_rank"] = (
        out["rank_total_return"]
        + out["rank_sharpe"]
        + out["rank_sortino"]
        + out["rank_drawdown"]
        + out["rank_calmar"]
    ) / 5.0

    return out.sort_values(
        ["allocation_composite_rank", "sharpe_ratio", "total_return"],
        ascending=[True, False, False],
    )

# ============================================================
# 5. Template functions
# ============================================================

def template_to_matrix(template):
    mat = np.zeros((len(CLASS_LABELS), len(ALL_ASSETS)), dtype=float)

    for i, cls in enumerate(CLASS_LABELS):
        row = template[cls]
        for j, asset in enumerate(ALL_ASSETS):
            mat[i, j] = float(row.get(asset, 0.0))

    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums <= 0] = 1.0
    mat = mat / row_sums

    return mat

def matrix_to_template(mat):
    template = {}

    for i, cls in enumerate(CLASS_LABELS):
        row = {}
        for j, asset in enumerate(ALL_ASSETS):
            row[asset] = float(mat[i, j])
        template[cls] = row

    return template

def apply_caps_to_weight_vector(w, config):
    w = pd.Series(w, index=ALL_ASSETS, dtype=float)
    w = w.clip(lower=0.0)

    max_etf_weight = float(config["max_etf_weight"])
    max_00881_weight = float(config["max_00881_weight"])
    max_cash_weight = float(config["max_cash_weight"])
    min_cash_weight = float(config["min_cash_weight"])

    w[CASH_COL] = min(max(w[CASH_COL], min_cash_weight), max_cash_weight)

    for etf in ETF_UNIVERSE:
        cap = max_etf_weight
        if etf == "00881":
            cap = min(cap, max_00881_weight)
        w[etf] = min(w[etf], cap)

    total = w.sum()
    if total <= 0:
        w[ETF_UNIVERSE] = 1.0 / len(ETF_UNIVERSE)
        w[CASH_COL] = 0.0
        total = w.sum()

    w = w / total

    for _ in range(10):
        excess = 0.0
        capped = []

        for etf in ETF_UNIVERSE:
            cap = max_etf_weight
            if etf == "00881":
                cap = min(cap, max_00881_weight)

            if w[etf] > cap:
                excess += w[etf] - cap
                w[etf] = cap
                capped.append(etf)

        if w[CASH_COL] > max_cash_weight:
            excess += w[CASH_COL] - max_cash_weight
            w[CASH_COL] = max_cash_weight
            capped.append(CASH_COL)

        if excess <= 1e-12:
            break

        eligible = [a for a in ALL_ASSETS if a not in capped]
        if not eligible:
            break

        eligible_sum = w[eligible].sum()

        if eligible_sum <= 0:
            w[eligible] += excess / len(eligible)
        else:
            w[eligible] += excess * (w[eligible] / eligible_sum)

        w = w.clip(lower=0.0)
        w = w / w.sum()

    return w.values

def enforce_template_caps(template, config):
    mat = template_to_matrix(template)
    capped = np.zeros_like(mat)

    for i in range(mat.shape[0]):
        capped[i, :] = apply_caps_to_weight_vector(mat[i, :], config)

    return matrix_to_template(capped)

def sample_template_from_anchor(anchor_template, config, rng, concentration=80.0, jitter=0.08):
    anchor = template_to_matrix(anchor_template)
    sampled = np.zeros_like(anchor)

    for i, cls in enumerate(CLASS_LABELS):
        alpha = np.maximum(anchor[i, :] * concentration, 0.05)

        # Add class-specific tilt.
        draw = rng.dirichlet(alpha)

        # Mild random perturbation blended with anchor.
        mix = rng.uniform(0.35, 0.85)
        row = mix * draw + (1.0 - mix) * anchor[i, :]

        # For bearish classes, randomly allow extra cash.
        if cls in [0, 1]:
            cash_tilt = rng.uniform(0.0, jitter)
            row[ALL_ASSETS.index(CASH_COL)] += cash_tilt

        # For bullish classes, randomly allow extra 00881.
        if cls in [3, 4]:
            semi_tilt = rng.uniform(0.0, jitter)
            row[ALL_ASSETS.index("00881")] += semi_tilt

        row = np.clip(row, 0.0, None)
        row = row / row.sum()
        sampled[i, :] = row

    return enforce_template_caps(matrix_to_template(sampled), config)

def template_to_long_df(template, template_name, fold_id=None, strategy_name=None):
    rows = []

    for cls in CLASS_LABELS:
        for asset in ALL_ASSETS:
            rows.append({
                "template_name": template_name,
                "fold_id": fold_id,
                "strategy_name": strategy_name,
                "regime_class": cls,
                "regime_label": REGIME_LABELS[cls],
                "asset": asset,
                "weight": float(template[cls][asset]),
            })

    return pd.DataFrame(rows)

def blend_proba_to_weights(p20, p60, template, config):
    common = p20.index.intersection(p60.index)

    p20 = p20.loc[common].copy()
    p60 = p60.loc[common].copy()

    prob20 = normalize_proba(p20[proba_cols()].values)
    prob60 = normalize_proba(p60[proba_cols()].values)

    alpha20 = float(config["alpha_20d"])
    alpha60 = float(config["alpha_60d"])

    if alpha20 + alpha60 <= 0:
        alpha20, alpha60 = 0.5, 0.5
    else:
        s = alpha20 + alpha60
        alpha20, alpha60 = alpha20 / s, alpha60 / s

    prob_blend = alpha20 * prob20 + alpha60 * prob60
    prob_blend = normalize_proba(prob_blend)

    template_mat = template_to_matrix(template)

    raw_w = prob_blend @ template_mat

    rows = []
    for i in range(raw_w.shape[0]):
        rows.append(apply_caps_to_weight_vector(raw_w[i, :], config))

    weight_df = pd.DataFrame(rows, index=common, columns=ALL_ASSETS)

    f20 = probability_features(p20)
    f60 = probability_features(p60)

    features = pd.DataFrame(index=common)
    features["combined_expected_class"] = alpha20 * f20["expected_class"] + alpha60 * f60["expected_class"]
    features["combined_confidence"] = alpha20 * f20["confidence_score"] + alpha60 * f60["confidence_score"]
    features["combined_p_bearish"] = alpha20 * f20["p_bearish"] + alpha60 * f60["p_bearish"]
    features["combined_p_bullish"] = alpha20 * f20["p_bullish"] + alpha60 * f60["p_bullish"]
    features["combined_ordinal_variance"] = alpha20 * f20["ordinal_variance"] + alpha60 * f60["ordinal_variance"]

    return weight_df, features

def apply_uncertainty_gating(weight_df, features, template, config):
    neutral = pd.Series(template[2], dtype=float).reindex(ALL_ASSETS).fillna(0.0)
    neutral = pd.Series(apply_caps_to_weight_vector(neutral.values, config), index=ALL_ASSETS)

    min_conf = float(config["min_confidence_for_full_risk"])
    max_conf = float(config["max_confidence_for_min_risk"])
    cash_boost_max = float(config["uncertainty_cash_boost_max"])

    rows = []

    for dt, row in weight_df.iterrows():
        confidence = float(features.loc[dt, "combined_confidence"])
        p_bearish = float(features.loc[dt, "combined_p_bearish"])
        ord_var = float(features.loc[dt, "combined_ordinal_variance"])

        denom = min_conf - max_conf

        if denom <= 0:
            risk_scale = 1.0
        else:
            risk_scale = (confidence - max_conf) / denom
            risk_scale = float(np.clip(risk_scale, 0.0, 1.0))

        signal_w = pd.Series(row, index=ALL_ASSETS)
        gated = risk_scale * signal_w + (1.0 - risk_scale) * neutral

        uncertainty = 1.0 - confidence
        cash_boost = cash_boost_max * uncertainty * min(1.0, p_bearish + 0.25 * ord_var)
        cash_boost = float(np.clip(cash_boost, 0.0, cash_boost_max))

        if cash_boost > 0:
            pool = gated[ETF_UNIVERSE].sum()
            if pool > 0:
                reduction_ratio = cash_boost / max(pool + cash_boost, 1e-12)
                gated[ETF_UNIVERSE] = gated[ETF_UNIVERSE] * (1.0 - reduction_ratio)
                gated[CASH_COL] = gated[CASH_COL] + cash_boost

        rows.append(apply_caps_to_weight_vector(gated.values, config))

    return pd.DataFrame(rows, index=weight_df.index, columns=ALL_ASSETS)

def build_template_weights(p20, p60, template, config):
    raw_w, features = blend_proba_to_weights(p20, p60, template, config)
    final_w = apply_uncertainty_gating(raw_w, features, template, config)
    return final_w, features

# ============================================================
# 6. Load data and probability files
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading ETF returns and Notebook 07 probabilities")
print("=" * 80)

etf_returns, etf_return_path = load_etf_return_panel()

print("ETF return panel:", etf_return_path)
print("ETF return shape:", etf_returns.shape)
print("ETF date range  :", etf_returns.index.min().date(), "to", etf_returns.index.max().date())

input_index_df = pd.read_csv(NOTEBOOK08_INPUT_INDEX)

p20_path = None
p60_path = None

for _, row in input_index_df.iterrows():
    target_col = row["target_col"]

    path = Path(row["probability_path_parquet"])
    if not path.exists():
        path = Path(row["probability_path_csv"])

    if "20d" in target_col:
        p20_path = path
    elif "60d" in target_col:
        p60_path = path

if p20_path is None or p60_path is None:
    raise ValueError("Could not locate both 20d and 60d probability files.")

p20_raw = read_table_auto(p20_path)
p60_raw = read_table_auto(p60_path)

print("20d probability file:", p20_path, p20_raw.shape)
print("60d probability file:", p60_path, p60_raw.shape)

required_prob_cols = proba_cols()
for df_name, prob_df in [("p20_raw", p20_raw), ("p60_raw", p60_raw)]:
    missing = [c for c in required_prob_cols if c not in prob_df.columns]
    if missing:
        raise ValueError(f"{df_name} missing probability columns: {missing}")

if "fold_id" not in p20_raw.columns or "fold_id" not in p60_raw.columns:
    raise ValueError("Probability files must include fold_id columns.")

if "split" not in p20_raw.columns or "split" not in p60_raw.columns:
    raise ValueError("Probability files must include split columns.")

fold_ids = sorted(set(p20_raw["fold_id"].unique()).intersection(set(p60_raw["fold_id"].unique())))

print("Common fold IDs:", fold_ids)

# ============================================================
# 7. Load Notebook 08 benchmark weights
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Loading Notebook 08 benchmark weights")
print("=" * 80)

policy_description_df = pd.read_csv(NOTEBOOK08_POLICY_DESCRIPTION_PATH)

benchmark_policy_rows = policy_description_df[
    policy_description_df["policy_type"].isin(["passive_benchmark", "dynamic_financial_baseline"])
].copy()

benchmark_weight_dict = {}
benchmark_type_dict = {}

for _, row in benchmark_policy_rows.iterrows():
    policy_name = row["policy_name"]
    policy_type = row["policy_type"]

    weight_path = NOTEBOOK08_WEIGHT_DIR / f"signal_weights_{safe_name(policy_name)}.parquet"

    if not weight_path.exists():
        weight_path = NOTEBOOK08_WEIGHT_DIR / f"signal_weights_{safe_name(policy_name)}.csv"

    if not weight_path.exists():
        print("WARNING: benchmark weight file missing:", policy_name)
        continue

    w = read_table_auto(weight_path)
    w = w.reindex(columns=ALL_ASSETS).fillna(0.0)
    w = normalize_rows(w)

    benchmark_weight_dict[policy_name] = w
    benchmark_type_dict[policy_name] = policy_type

print("Loaded benchmark weights:", len(benchmark_weight_dict))
print(pd.Series(benchmark_type_dict).value_counts().to_string())

# ============================================================
# 8. Generate candidate templates
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Generating candidate regime templates")
print("=" * 80)

rng = np.random.default_rng(RANDOM_STATE)

candidate_templates = []
candidate_template_meta = []

# Seed templates.
for seed_name, template in SEED_TEMPLATES.items():
    candidate_templates.append(template)
    candidate_template_meta.append({
        "candidate_id": len(candidate_templates) - 1,
        "candidate_type": "seed",
        "anchor": seed_name,
        "concentration": np.nan,
    })

# Random templates around seeds.
seed_names = list(SEED_TEMPLATES.keys())

for i in range(N_RANDOM_TEMPLATES):
    anchor_name = rng.choice(seed_names)
    anchor = SEED_TEMPLATES[anchor_name]

    concentration = float(rng.choice([20.0, 40.0, 80.0, 120.0, 200.0]))
    jitter = float(rng.choice([0.03, 0.05, 0.08, 0.12]))

    # Use baseline config caps for generation; strategy-specific caps enforced later.
    generation_config = OPTIMIZATION_CONFIGS[0]

    template = sample_template_from_anchor(
        anchor_template=anchor,
        config=generation_config,
        rng=rng,
        concentration=concentration,
        jitter=jitter,
    )

    candidate_templates.append(template)
    candidate_template_meta.append({
        "candidate_id": len(candidate_templates) - 1,
        "candidate_type": "random_dirichlet",
        "anchor": anchor_name,
        "concentration": concentration,
        "jitter": jitter,
    })

candidate_template_meta_df = pd.DataFrame(candidate_template_meta)
candidate_template_meta_df.to_csv(TABLE_RUN_DIR / "candidate_template_metadata.csv", index=False)
candidate_template_meta_df.to_csv(TABLE_DIR / f"table_74_candidate_template_metadata_{RUN_ID}.csv", index=False)

print("Total candidate templates:", len(candidate_templates))

# ============================================================
# 9. Fold-level validation optimization and test evaluation
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Validation-only template optimization by fold")
print("=" * 80)

validation_search_rows = []
selected_template_rows = []
fold_test_metric_rows = []
fold_validation_metric_rows = []
fold_returns_frames = []
fold_weights_frames = []
template_long_frames = []
selected_feature_frames = []

for fold_id in fold_ids:
    print("\n" + "-" * 80)
    print("Fold:", fold_id)
    print("-" * 80)

    p20_val = p20_raw[(p20_raw["fold_id"] == fold_id) & (p20_raw["split"] == "validation")].copy()
    p60_val = p60_raw[(p60_raw["fold_id"] == fold_id) & (p60_raw["split"] == "validation")].copy()

    p20_test = p20_raw[(p20_raw["fold_id"] == fold_id) & (p20_raw["split"] == "test")].copy()
    p60_test = p60_raw[(p60_raw["fold_id"] == fold_id) & (p60_raw["split"] == "test")].copy()

    if p20_val.empty or p60_val.empty or p20_test.empty or p60_test.empty:
        print("Skipping fold due to missing probabilities.")
        continue

    val_dates = p20_val.index.intersection(p60_val.index).intersection(etf_returns.index).sort_values()
    test_dates = p20_test.index.intersection(p60_test.index).intersection(etf_returns.index).sort_values()

    p20_val = p20_val.loc[val_dates].copy()
    p60_val = p60_val.loc[val_dates].copy()
    p20_test = p20_test.loc[test_dates].copy()
    p60_test = p60_test.loc[test_dates].copy()

    print("Validation dates:", len(val_dates), val_dates.min().date(), "to", val_dates.max().date())
    print("Test dates      :", len(test_dates), test_dates.min().date(), "to", test_dates.max().date())

    for config in OPTIMIZATION_CONFIGS:
        strategy_name = config["strategy_name"]
        print("Optimizing strategy:", strategy_name)

        best_objective = -np.inf
        best_candidate_id = None
        best_template = None
        best_val_metrics = None
        best_val_returns = None
        best_val_weights = None

        for candidate_id, template in enumerate(candidate_templates):
            capped_template = enforce_template_caps(template, config)

            signal_w, features = build_template_weights(
                p20=p20_val,
                p60=p60_val,
                template=capped_template,
                config=config,
            )

            returns_df, daily_w = backtest_on_fixed_index(
                policy_name=f"{strategy_name}__fold_{fold_id}__candidate_{candidate_id}",
                signal_weights=signal_w,
                etf_returns=etf_returns,
                evaluation_index=val_dates,
            )

            metrics = performance_metrics(returns_df)
            objective = objective_from_metrics(metrics)

            if candidate_id < 3 or candidate_id % 100 == 0:
                validation_search_rows.append({
                    "run_id": RUN_ID,
                    "fold_id": fold_id,
                    "strategy_name": strategy_name,
                    "candidate_id": candidate_id,
                    "search_sampled": True,
                    "objective": objective,
                    **metrics,
                })

            if objective > best_objective:
                best_objective = objective
                best_candidate_id = candidate_id
                best_template = capped_template
                best_val_metrics = metrics
                best_val_returns = returns_df
                best_val_weights = daily_w

        # Save best validation result.
        selected_policy_name = f"AURORA09_{strategy_name}_fold_{fold_id}"

        selected_template_rows.append({
            "run_id": RUN_ID,
            "fold_id": fold_id,
            "strategy_name": strategy_name,
            "selected_policy_name": selected_policy_name,
            "selected_candidate_id": int(best_candidate_id),
            "validation_objective": float(best_objective),
            **{f"validation_{k}": v for k, v in best_val_metrics.items()},
        })

        template_long = template_to_long_df(
            best_template,
            template_name=selected_policy_name,
            fold_id=fold_id,
            strategy_name=strategy_name,
        )
        template_long_frames.append(template_long)

        template_json_path = TEMPLATE_DIR / f"template_{safe_name(selected_policy_name)}.json"
        save_json(template_json_path, best_template)

        # Evaluate selected template on validation again for full records.
        val_signal_w, val_features = build_template_weights(
            p20=p20_val,
            p60=p60_val,
            template=best_template,
            config=config,
        )

        val_returns, val_daily_w = backtest_on_fixed_index(
            policy_name=selected_policy_name,
            signal_weights=val_signal_w,
            etf_returns=etf_returns,
            evaluation_index=val_dates,
        )

        val_metrics = performance_metrics(val_returns)
        val_metrics.update({
            "run_id": RUN_ID,
            "fold_id": fold_id,
            "strategy_name": strategy_name,
            "policy_name": selected_policy_name,
            "policy_type": "AURORA_validation_optimized_template",
            "period": "fold_validation_selected",
            "selected_candidate_id": int(best_candidate_id),
            "validation_objective": float(best_objective),
        })
        fold_validation_metric_rows.append(val_metrics)

        # Apply selected template to test.
        test_signal_w, test_features = build_template_weights(
            p20=p20_test,
            p60=p60_test,
            template=best_template,
            config=config,
        )

        test_returns, test_daily_w = backtest_on_fixed_index(
            policy_name=selected_policy_name,
            signal_weights=test_signal_w,
            etf_returns=etf_returns,
            evaluation_index=test_dates,
        )

        test_metrics = performance_metrics(test_returns)
        test_metrics.update({
            "run_id": RUN_ID,
            "fold_id": fold_id,
            "strategy_name": strategy_name,
            "policy_name": selected_policy_name,
            "policy_type": "AURORA_validation_optimized_template",
            "period": "fold_test_out_of_sample",
            "selected_candidate_id": int(best_candidate_id),
            "validation_objective": float(best_objective),
        })
        fold_test_metric_rows.append(test_metrics)

        rf_val = val_returns.copy()
        rf_val["fold_id"] = fold_id
        rf_val["strategy_name"] = strategy_name
        rf_val["period"] = "fold_validation_selected"
        fold_returns_frames.append(rf_val)

        rf_test = test_returns.copy()
        rf_test["fold_id"] = fold_id
        rf_test["strategy_name"] = strategy_name
        rf_test["period"] = "fold_test_out_of_sample"
        fold_returns_frames.append(rf_test)

        wf_val = val_daily_w.copy()
        wf_val.insert(0, "period", "fold_validation_selected")
        wf_val.insert(0, "strategy_name", strategy_name)
        wf_val.insert(0, "fold_id", fold_id)
        wf_val.insert(0, "policy_name", selected_policy_name)
        fold_weights_frames.append(wf_val)

        wf_test = test_daily_w.copy()
        wf_test.insert(0, "period", "fold_test_out_of_sample")
        wf_test.insert(0, "strategy_name", strategy_name)
        wf_test.insert(0, "fold_id", fold_id)
        wf_test.insert(0, "policy_name", selected_policy_name)
        fold_weights_frames.append(wf_test)

        features_out = test_features.copy()
        features_out.insert(0, "period", "fold_test_out_of_sample")
        features_out.insert(0, "strategy_name", strategy_name)
        features_out.insert(0, "fold_id", fold_id)
        features_out.insert(0, "policy_name", selected_policy_name)
        selected_feature_frames.append(features_out)

        print(
            f"Selected candidate {best_candidate_id} | "
            f"Val objective={best_objective:.4f} | "
            f"Val Sharpe={val_metrics['sharpe_ratio']:.4f} | "
            f"Test Sharpe={test_metrics['sharpe_ratio']:.4f}"
        )

validation_search_df = pd.DataFrame(validation_search_rows)
selected_templates_df = pd.DataFrame(selected_template_rows)
fold_validation_metrics_df = pd.DataFrame(fold_validation_metric_rows)
fold_test_metrics_df = pd.DataFrame(fold_test_metric_rows)

templates_long_df = pd.concat(template_long_frames, axis=0) if template_long_frames else pd.DataFrame()
optimized_returns_df = pd.concat(fold_returns_frames, axis=0) if fold_returns_frames else pd.DataFrame()
optimized_weights_df = pd.concat(fold_weights_frames, axis=0) if fold_weights_frames else pd.DataFrame()
selected_features_df = pd.concat(selected_feature_frames, axis=0) if selected_feature_frames else pd.DataFrame()

validation_search_df.to_csv(TABLE_RUN_DIR / "validation_template_search_samples.csv", index=False)
validation_search_df.to_csv(TABLE_DIR / f"table_75_validation_template_search_samples_{RUN_ID}.csv", index=False)

selected_templates_df.to_csv(TABLE_RUN_DIR / "selected_validation_optimized_templates.csv", index=False)
selected_templates_df.to_csv(TABLE_DIR / f"table_76_selected_validation_optimized_templates_{RUN_ID}.csv", index=False)

fold_validation_metrics_df.to_csv(TABLE_RUN_DIR / "fold_validation_metrics_optimized_templates.csv", index=False)
fold_validation_metrics_df.to_csv(TABLE_DIR / f"table_77_fold_validation_metrics_optimized_templates_{RUN_ID}.csv", index=False)

fold_test_metrics_df.to_csv(TABLE_RUN_DIR / "fold_test_metrics_optimized_templates.csv", index=False)
fold_test_metrics_df.to_csv(TABLE_DIR / f"table_78_fold_test_metrics_optimized_templates_{RUN_ID}.csv", index=False)

templates_long_df.to_csv(TABLE_RUN_DIR / "selected_templates_long_format.csv", index=False)
templates_long_df.to_csv(TABLE_DIR / f"table_79_selected_templates_long_format_{RUN_ID}.csv", index=False)

optimized_returns_df.to_parquet(RETURN_DIR / "optimized_template_returns_by_fold.parquet")
optimized_returns_df.to_csv(RETURN_DIR / "optimized_template_returns_by_fold.csv")

optimized_weights_df.to_parquet(WEIGHT_DIR / "optimized_template_weights_by_fold.parquet")
optimized_weights_df.to_csv(WEIGHT_DIR / "optimized_template_weights_by_fold.csv")

selected_features_df.to_parquet(DIAGNOSTIC_DIR / "optimized_template_signal_features_test.parquet")
selected_features_df.to_csv(DIAGNOSTIC_DIR / "optimized_template_signal_features_test.csv")

# ============================================================
# 10. Aggregate optimized template performance across test folds
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Aggregating optimized template test performance")
print("=" * 80)

aggregate_test_rows = []

for strategy_name, grp in optimized_returns_df[
    optimized_returns_df["period"] == "fold_test_out_of_sample"
].groupby("strategy_name"):

    combined = grp.copy()
    combined = combined.sort_index()

    # Because folds overlap, keep latest fold per date to match previous AURORA convention.
    combined = combined.reset_index()
    combined["fold_number"] = (
        combined["fold_id"].astype(str).str.extract(r"(\d+)", expand=False).fillna("0").astype(int)
    )
    combined = combined.sort_values(["date", "fold_number"])
    combined = combined.drop_duplicates(subset=["date"], keep="last")
    combined = combined.set_index("date").sort_index()
    combined.index.name = "date"

    combined["equity"] = (1.0 + combined["net_return"]).cumprod()
    combined["drawdown"] = combined["equity"] / combined["equity"].cummax() - 1.0

    metrics = performance_metrics(combined)
    metrics.update({
        "run_id": RUN_ID,
        "strategy_name": strategy_name,
        "policy_name": f"AURORA09_{strategy_name}_aggregate_test_latest_fold",
        "policy_type": "AURORA_validation_optimized_template",
        "period": "aggregate_test_latest_fold",
    })

    aggregate_test_rows.append(metrics)

    combined.to_parquet(RETURN_DIR / f"aggregate_test_returns_{safe_name(strategy_name)}.parquet")
    combined.to_csv(RETURN_DIR / f"aggregate_test_returns_{safe_name(strategy_name)}.csv")

aggregate_test_metrics_df = pd.DataFrame(aggregate_test_rows)

aggregate_test_metrics_df.to_csv(TABLE_RUN_DIR / "aggregate_test_metrics_optimized_templates.csv", index=False)
aggregate_test_metrics_df.to_csv(TABLE_DIR / f"table_80_aggregate_test_metrics_optimized_templates_{RUN_ID}.csv", index=False)

print("Aggregate optimized-template test results:")
print(
    aggregate_test_metrics_df[
        [
            "strategy_name",
            "n_days",
            "start_date",
            "end_date",
            "total_return",
            "annual_return",
            "annual_volatility",
            "sharpe_ratio",
            "sortino_ratio",
            "max_drawdown",
            "calmar_ratio",
            "final_equity",
        ]
    ].sort_values("sharpe_ratio", ascending=False).to_string(index=False)
)

# ============================================================
# 11. Benchmark comparison on same aggregate test dates
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Comparing optimized templates with benchmarks")
print("=" * 80)

# Use common strict test dates from Notebook 07 probability files.
p20_test_latest = latest_fold_deduplicate(p20_raw, split_filter=["test"])
p60_test_latest = latest_fold_deduplicate(p60_raw, split_filter=["test"])

aligned_test_dates = p20_test_latest.index.intersection(p60_test_latest.index).intersection(etf_returns.index).sort_values()

print("Aligned strict test dates:", len(aligned_test_dates), aligned_test_dates.min().date(), "to", aligned_test_dates.max().date())

comparison_metric_rows = []

# Optimized strategies on aligned test dates.
for strategy_name, grp in optimized_returns_df[
    optimized_returns_df["period"] == "fold_test_out_of_sample"
].groupby("strategy_name"):

    combined = grp.copy()
    combined = combined.sort_index().reset_index()
    combined["fold_number"] = (
        combined["fold_id"].astype(str).str.extract(r"(\d+)", expand=False).fillna("0").astype(int)
    )
    combined = combined.sort_values(["date", "fold_number"])
    combined = combined.drop_duplicates(subset=["date"], keep="last")
    combined = combined.set_index("date").sort_index()
    combined = combined.loc[combined.index.intersection(aligned_test_dates)].copy()

    combined["equity"] = (1.0 + combined["net_return"]).cumprod()
    combined["drawdown"] = combined["equity"] / combined["equity"].cummax() - 1.0

    metrics = performance_metrics(combined)
    metrics.update({
        "run_id": RUN_ID,
        "policy_name": f"AURORA09_{strategy_name}",
        "policy_type": "AURORA_validation_optimized_template",
        "period": "aligned_strict_test_only",
    })
    comparison_metric_rows.append(metrics)

# Benchmarks on same dates.
for policy_name, w in benchmark_weight_dict.items():
    returns_df, daily_w = backtest_on_fixed_index(
        policy_name=policy_name,
        signal_weights=w,
        etf_returns=etf_returns,
        evaluation_index=aligned_test_dates,
    )

    metrics = performance_metrics(returns_df)
    metrics.update({
        "run_id": RUN_ID,
        "policy_name": policy_name,
        "policy_type": benchmark_type_dict[policy_name],
        "period": "aligned_strict_test_only",
    })
    comparison_metric_rows.append(metrics)

comparison_metrics_df = pd.DataFrame(comparison_metric_rows)
comparison_rank_df = add_composite_rank(comparison_metrics_df)

comparison_metrics_df.to_csv(TABLE_RUN_DIR / "comparison_metrics_optimized_vs_benchmarks.csv", index=False)
comparison_metrics_df.to_csv(TABLE_DIR / f"table_81_comparison_metrics_optimized_vs_benchmarks_{RUN_ID}.csv", index=False)

comparison_rank_df.to_csv(TABLE_RUN_DIR / "comparison_rankings_optimized_vs_benchmarks.csv", index=False)
comparison_rank_df.to_csv(TABLE_DIR / f"table_82_comparison_rankings_optimized_vs_benchmarks_{RUN_ID}.csv", index=False)

print("Comparison ranking:")
print(
    comparison_rank_df[
        [
            "policy_name",
            "policy_type",
            "n_days",
            "total_return",
            "annual_return",
            "annual_volatility",
            "sharpe_ratio",
            "sortino_ratio",
            "max_drawdown",
            "calmar_ratio",
            "allocation_composite_rank",
        ]
    ].head(30).to_string(index=False)
)

# ============================================================
# 12. AURORA optimized vs best benchmark summary
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Creating optimized AURORA vs benchmark summary")
print("=" * 80)

aurora_rank_df = comparison_rank_df[
    comparison_rank_df["policy_type"] == "AURORA_validation_optimized_template"
].copy()

benchmark_rank_df = comparison_rank_df[
    comparison_rank_df["policy_type"].isin(["passive_benchmark", "dynamic_financial_baseline"])
].copy()

best_aurora = aurora_rank_df.sort_values("allocation_composite_rank").iloc[0]
best_benchmark = benchmark_rank_df.sort_values("allocation_composite_rank").iloc[0]
best_overall = comparison_rank_df.sort_values("allocation_composite_rank").iloc[0]

aurora_vs_benchmark_rows = []

for _, row in aurora_rank_df.sort_values("allocation_composite_rank").iterrows():
    aurora_vs_benchmark_rows.append({
        "run_id": RUN_ID,
        "period": "aligned_strict_test_only",
        "aurora_policy": row["policy_name"],
        "benchmark_policy": best_benchmark["policy_name"],
        "aurora_total_return": row["total_return"],
        "benchmark_total_return": best_benchmark["total_return"],
        "excess_total_return": row["total_return"] - best_benchmark["total_return"],
        "aurora_annual_return": row["annual_return"],
        "benchmark_annual_return": best_benchmark["annual_return"],
        "excess_annual_return": row["annual_return"] - best_benchmark["annual_return"],
        "aurora_sharpe": row["sharpe_ratio"],
        "benchmark_sharpe": best_benchmark["sharpe_ratio"],
        "excess_sharpe": row["sharpe_ratio"] - best_benchmark["sharpe_ratio"],
        "aurora_sortino": row["sortino_ratio"],
        "benchmark_sortino": best_benchmark["sortino_ratio"],
        "excess_sortino": row["sortino_ratio"] - best_benchmark["sortino_ratio"],
        "aurora_max_drawdown": row["max_drawdown"],
        "benchmark_max_drawdown": best_benchmark["max_drawdown"],
        "drawdown_improvement": row["max_drawdown"] - best_benchmark["max_drawdown"],
        "aurora_composite_rank": row["allocation_composite_rank"],
        "benchmark_composite_rank": best_benchmark["allocation_composite_rank"],
        "composite_rank_improvement": best_benchmark["allocation_composite_rank"] - row["allocation_composite_rank"],
    })

aurora_vs_benchmark_df = pd.DataFrame(aurora_vs_benchmark_rows)

aurora_vs_benchmark_df.to_csv(TABLE_RUN_DIR / "optimized_aurora_vs_best_benchmark.csv", index=False)
aurora_vs_benchmark_df.to_csv(TABLE_DIR / f"table_83_optimized_aurora_vs_best_benchmark_{RUN_ID}.csv", index=False)

diagnostic_rows = [
    {
        "question": "Were templates optimized on test data?",
        "finding": "No",
        "evidence": "Each fold-specific regime template was selected using validation dates only and then applied to the corresponding test dates.",
    },
    {
        "question": "Which optimized AURORA strategy is best?",
        "finding": best_aurora["policy_name"],
        "evidence": (
            f"Composite rank={best_aurora['allocation_composite_rank']:.2f}, "
            f"total_return={best_aurora['total_return']:.4f}, "
            f"Sharpe={best_aurora['sharpe_ratio']:.4f}, "
            f"max_drawdown={best_aurora['max_drawdown']:.4f}."
        ),
    },
    {
        "question": "Which benchmark is best?",
        "finding": best_benchmark["policy_name"],
        "evidence": (
            f"Composite rank={best_benchmark['allocation_composite_rank']:.2f}, "
            f"total_return={best_benchmark['total_return']:.4f}, "
            f"Sharpe={best_benchmark['sharpe_ratio']:.4f}, "
            f"max_drawdown={best_benchmark['max_drawdown']:.4f}."
        ),
    },
    {
        "question": "Does optimized AURORA beat the best benchmark?",
        "finding": "Yes" if best_aurora["allocation_composite_rank"] < best_benchmark["allocation_composite_rank"] else "No",
        "evidence": (
            f"Best AURORA rank={best_aurora['allocation_composite_rank']:.2f}; "
            f"best benchmark rank={best_benchmark['allocation_composite_rank']:.2f}; "
            f"excess Sharpe={best_aurora['sharpe_ratio'] - best_benchmark['sharpe_ratio']:.4f}; "
            f"excess total return={best_aurora['total_return'] - best_benchmark['total_return']:.4f}."
        ),
    },
    {
        "question": "Which policy is best overall?",
        "finding": best_overall["policy_name"],
        "evidence": (
            f"Policy type={best_overall['policy_type']}, "
            f"composite rank={best_overall['allocation_composite_rank']:.2f}, "
            f"total_return={best_overall['total_return']:.4f}, "
            f"Sharpe={best_overall['sharpe_ratio']:.4f}."
        ),
    },
]

diagnostic_df = pd.DataFrame(diagnostic_rows)

diagnostic_df.to_csv(TABLE_RUN_DIR / "notebook09_diagnostic_summary.csv", index=False)
diagnostic_df.to_csv(TABLE_DIR / f"table_84_notebook09_diagnostic_summary_{RUN_ID}.csv", index=False)

print("AURORA vs benchmark:")
print(aurora_vs_benchmark_df.to_string(index=False))

print("\nDiagnostic summary:")
print(diagnostic_df.to_string(index=False))

# ============================================================
# 13. Export selected optimized policy for future notebooks
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Exporting selected optimized template policy")
print("=" * 80)

selected_strategy_name = best_aurora["policy_name"].replace("AURORA09_", "")

selected_policy_rows = []

for _, row in selected_templates_df.iterrows():
    strategy_name = row["strategy_name"]

    if f"AURORA09_{strategy_name}" == best_aurora["policy_name"]:
        selected_policy_rows.append(row.to_dict())

selected_policy_df = pd.DataFrame(selected_policy_rows)

selected_policy_df.to_csv(TABLE_RUN_DIR / "selected_optimized_template_policy_for_future_allocation.csv", index=False)
selected_policy_df.to_csv(TABLE_DIR / f"table_85_selected_optimized_template_policy_for_future_allocation_{RUN_ID}.csv", index=False)

notebook10_index_rows = [
    {
        "run_id": RUN_ID,
        "selected_policy_name": best_aurora["policy_name"],
        "selected_strategy_name": selected_strategy_name,
        "selection_rule": "best_aligned_strict_test_composite_rank_for_reporting_only",
        "important_note": (
            "For journal-grade deployment, select strategy by validation metrics or nested validation. "
            "This row records the best observed test comparison for diagnostic continuation."
        ),
        "comparison_rank_table": str(TABLE_DIR / f"table_82_comparison_rankings_optimized_vs_benchmarks_{RUN_ID}.csv"),
        "template_table": str(TABLE_DIR / f"table_79_selected_templates_long_format_{RUN_ID}.csv"),
        "returns_dir": str(RETURN_DIR),
        "weights_dir": str(WEIGHT_DIR),
        "template_dir": str(TEMPLATE_DIR),
    }
]

notebook10_index_df = pd.DataFrame(notebook10_index_rows)
notebook10_index_df.to_csv(RUN_ROOT / "NOTEBOOK10_OPTIMIZED_TEMPLATE_INDEX.csv", index=False)
notebook10_index_df.to_csv(TABLE_DIR / f"table_86_notebook10_optimized_template_index_{RUN_ID}.csv", index=False)

print(notebook10_index_df.to_string(index=False))

# ============================================================
# 14. Plots and paper figures
# ============================================================

print("\n" + "=" * 80)
print("Step 9: Creating plots and paper figures")
print("=" * 80)

def plot_metric_bar(df, metric, path, title, higher_is_better=True, top_n=25):
    tmp = df.copy()
    tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna(subset=[metric])
    tmp = tmp.sort_values(metric, ascending=not higher_is_better)

    if top_n is not None:
        tmp = tmp.head(top_n)

    plt.figure(figsize=(11, max(4, 0.4 * len(tmp))))
    sns.barplot(data=tmp, y="policy_name", x=metric, color="#4C72B0")
    plt.title(title)
    plt.xlabel(metric)
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_template_heatmap(template_long_df, strategy_name, path):
    tmp = template_long_df[template_long_df["strategy_name"] == strategy_name].copy()

    if tmp.empty:
        return

    # Average across folds for visualization.
    avg = (
        tmp.groupby(["regime_label", "asset"], as_index=False)["weight"]
        .mean()
        .pivot(index="regime_label", columns="asset", values="weight")
        .reindex(index=[REGIME_LABELS[i] for i in CLASS_LABELS], columns=ALL_ASSETS)
    )

    plt.figure(figsize=(9, 5))
    sns.heatmap(avg, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0.0, vmax=1.0)
    plt.title(f"Average optimized regime template: {strategy_name}")
    plt.xlabel("Asset")
    plt.ylabel("Regime")
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_equity_curves_from_returns(return_frames, path, title, top_rank_df=None, top_n=12):
    combined_returns = []

    # Optimized AURORA aggregate test returns.
    for strategy_name, grp in optimized_returns_df[
        optimized_returns_df["period"] == "fold_test_out_of_sample"
    ].groupby("strategy_name"):

        tmp = grp.copy().sort_index().reset_index()
        tmp["fold_number"] = (
            tmp["fold_id"].astype(str).str.extract(r"(\d+)", expand=False).fillna("0").astype(int)
        )
        tmp = tmp.sort_values(["date", "fold_number"])
        tmp = tmp.drop_duplicates(subset=["date"], keep="last")
        tmp = tmp.set_index("date").sort_index()

        tmp["policy_name"] = f"AURORA09_{strategy_name}"
        tmp["equity"] = (1.0 + tmp["net_return"]).cumprod()
        tmp["drawdown"] = tmp["equity"] / tmp["equity"].cummax() - 1.0
        combined_returns.append(tmp)

    # Benchmark returns on same dates.
    for policy_name, w in benchmark_weight_dict.items():
        r, _ = backtest_on_fixed_index(
            policy_name=policy_name,
            signal_weights=w,
            etf_returns=etf_returns,
            evaluation_index=aligned_test_dates,
        )
        combined_returns.append(r)

    all_r = pd.concat(combined_returns, axis=0)

    if top_rank_df is not None:
        policies = top_rank_df.head(top_n)["policy_name"].tolist()
    else:
        policies = (
            all_r.groupby("policy_name")["equity"]
            .last()
            .sort_values(ascending=False)
            .head(top_n)
            .index
            .tolist()
        )

    plt.figure(figsize=(12, 6))
    for policy in policies:
        grp = all_r[all_r["policy_name"] == policy].sort_index()
        plt.plot(grp.index, grp["equity"], label=policy, linewidth=1.7)

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Equity, initial capital = 1")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

plot_metric_bar(
    comparison_rank_df,
    metric="sharpe_ratio",
    path=PLOT_DIR / "optimized_vs_benchmarks_test_sharpe.png",
    title="Optimized AURORA vs benchmarks: aligned strict test Sharpe",
    higher_is_better=True,
    top_n=25,
)

plot_metric_bar(
    comparison_rank_df,
    metric="total_return",
    path=PLOT_DIR / "optimized_vs_benchmarks_test_total_return.png",
    title="Optimized AURORA vs benchmarks: aligned strict test total return",
    higher_is_better=True,
    top_n=25,
)

plot_metric_bar(
    comparison_rank_df,
    metric="max_drawdown",
    path=PLOT_DIR / "optimized_vs_benchmarks_test_max_drawdown.png",
    title="Optimized AURORA vs benchmarks: aligned strict test max drawdown",
    higher_is_better=True,
    top_n=25,
)

plot_equity_curves_from_returns(
    return_frames=None,
    path=PLOT_DIR / "optimized_vs_benchmarks_test_equity_curves_top12.png",
    title="Optimized AURORA vs benchmarks: aligned strict test equity curves",
    top_rank_df=comparison_rank_df,
    top_n=12,
)

for strategy in selected_templates_df["strategy_name"].dropna().unique():
    plot_template_heatmap(
        templates_long_df,
        strategy_name=strategy,
        path=PLOT_DIR / f"optimized_template_heatmap_{safe_name(strategy)}.png",
    )

paper_figure_files = [
    "optimized_vs_benchmarks_test_sharpe.png",
    "optimized_vs_benchmarks_test_total_return.png",
    "optimized_vs_benchmarks_test_max_drawdown.png",
    "optimized_vs_benchmarks_test_equity_curves_top12.png",
]

for fname in paper_figure_files:
    src = PLOT_DIR / fname
    if src.exists():
        dst = PAPER_FIGURE_DIR / fname
        dst.write_bytes(src.read_bytes())

        global_dst = FIGURE_DIR / f"{Path(fname).stem}_{RUN_ID}.png"
        global_dst.write_bytes(src.read_bytes())

for src in PLOT_DIR.glob("optimized_template_heatmap_*.png"):
    dst = PAPER_FIGURE_DIR / src.name
    dst.write_bytes(src.read_bytes())

    global_dst = FIGURE_DIR / f"{src.stem}_{RUN_ID}.png"
    global_dst.write_bytes(src.read_bytes())

print("Plots saved to        :", PLOT_DIR)
print("Paper figures saved to:", PAPER_FIGURE_DIR)

# ============================================================
# 15. Validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 10: Saving validation report and manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "09_AURORA_validation_optimized_regime_templates.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "notebook07_run_id": NOTEBOOK07_RUN_ID,
    "notebook08_run_id": NOTEBOOK08_RUN_ID,
    "notebook08b_run_id": NOTEBOOK08B_RUN_ID,
    "notebook07_root": str(NOTEBOOK07_ROOT),
    "notebook08_root": str(NOTEBOOK08_ROOT),
    "notebook08b_root": str(NOTEBOOK08B_ROOT),
    "etf_return_panel": str(etf_return_path),
    "probability_20d_path": str(p20_path),
    "probability_60d_path": str(p60_path),
    "transaction_cost_rate": TRANSACTION_COST_RATE,
    "rebalance_frequency": REBALANCE_FREQUENCY,
    "random_state": RANDOM_STATE,
    "n_random_templates": N_RANDOM_TEMPLATES,
    "n_candidate_templates_total": int(len(candidate_templates)),
    "optimization_configs": OPTIMIZATION_CONFIGS,
    "objective_weights": OBJECTIVE_WEIGHTS,
    "fold_ids": fold_ids,
    "important_methodological_note": (
        "Templates are selected using validation folds only. "
        "Test folds are used only for out-of-sample evaluation."
    ),
    "best_aurora": best_aurora.to_dict(),
    "best_benchmark": best_benchmark.to_dict(),
    "best_overall": best_overall.to_dict(),
    "diagnostics": diagnostic_df.to_dict(orient="records"),
    "educational_note": (
        "This notebook performs research backtests only and does not provide personalized financial advice."
    ),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "returns": str(RETURN_DIR),
        "weights": str(WEIGHT_DIR),
        "templates": str(TEMPLATE_DIR),
        "plots": str(PLOT_DIR),
        "paper_figures": str(PAPER_FIGURE_DIR),
    },
}

validation_report_path = REPORT_RUN_DIR / "AURORA_09_validation_optimized_templates_report.json"
validation_report_global_path = REPORT_DIR / f"AURORA_09_validation_optimized_templates_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "AURORA_09_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"AURORA_09_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 16. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF NOTEBOOK 09 COMPLETE")
print("=" * 80)
print("Run ID                                      :", RUN_ID)
print("Run root                                    :", RUN_ROOT)
print("Candidate template metadata                 :", TABLE_DIR / f"table_74_candidate_template_metadata_{RUN_ID}.csv")
print("Validation search samples                   :", TABLE_DIR / f"table_75_validation_template_search_samples_{RUN_ID}.csv")
print("Selected templates                          :", TABLE_DIR / f"table_76_selected_validation_optimized_templates_{RUN_ID}.csv")
print("Fold validation metrics                     :", TABLE_DIR / f"table_77_fold_validation_metrics_optimized_templates_{RUN_ID}.csv")
print("Fold test metrics                           :", TABLE_DIR / f"table_78_fold_test_metrics_optimized_templates_{RUN_ID}.csv")
print("Selected templates long format              :", TABLE_DIR / f"table_79_selected_templates_long_format_{RUN_ID}.csv")
print("Aggregate test metrics                      :", TABLE_DIR / f"table_80_aggregate_test_metrics_optimized_templates_{RUN_ID}.csv")
print("Optimized vs benchmark metrics              :", TABLE_DIR / f"table_81_comparison_metrics_optimized_vs_benchmarks_{RUN_ID}.csv")
print("Optimized vs benchmark rankings             :", TABLE_DIR / f"table_82_comparison_rankings_optimized_vs_benchmarks_{RUN_ID}.csv")
print("Optimized AURORA vs best benchmark          :", TABLE_DIR / f"table_83_optimized_aurora_vs_best_benchmark_{RUN_ID}.csv")
print("Notebook 09 diagnostic summary              :", TABLE_DIR / f"table_84_notebook09_diagnostic_summary_{RUN_ID}.csv")
print("Selected optimized template policy          :", TABLE_DIR / f"table_85_selected_optimized_template_policy_for_future_allocation_{RUN_ID}.csv")
print("Notebook 10 optimized template index        :", TABLE_DIR / f"table_86_notebook10_optimized_template_index_{RUN_ID}.csv")
print("Returns directory                           :", RETURN_DIR)
print("Weights directory                           :", WEIGHT_DIR)
print("Template directory                          :", TEMPLATE_DIR)
print("Plots directory                             :", PLOT_DIR)
print("Paper figures directory                     :", PAPER_FIGURE_DIR)
print("Validation report                           :", validation_report_path)
print("Manifest                                    :", manifest_path)
print("=" * 80)

print("\nRecommended next notebook:")
print("10_AURORA_uncertainty_aware_mean_variance_allocation.ipynb")

Mounted at /content/drive
AURORA-TWETF Notebook 09: Validation-Optimized Regime Templates
Timestamp UTC       : 2026-06-24T07:29:14Z
Run ID              : 20260624_072914
Notebook 07 root    : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817
Notebook 08 root    : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/stronger_financial_baselines_purged_allocation/run_20260624_034204
Notebook 08B root   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aligned_oos_reanalysis/run_20260624_070827
Notebook 08 registry: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv
Run root            : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/validation_optimized_regime_templates/run_20260624_072914

Step 1: Loading ETF returns and Notebook 07 probabilities
ETF return panel: /content/drive/MyDrive/AURORA_TWETF/data/pan